<a href="https://colab.research.google.com/github/mohamadatashfaraz4-netizen/master-thesis/blob/main/5_Maschinelles_Lernen_fuer_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
df = pd.read_excel("credit_card_corrupted_all_errors.xlsx")

# Spaltennamen standardisieren
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

df.head()

Saving credit_card_corrupted_all_errors.xlsx to credit_card_corrupted_all_errors.xlsx


,id,limit_bal,sex,education,marriage,age,pay_0,pay_2,pay_3,pay_4,...,bill_amt4,bill_amt5,bill_amt6,pay_amt1,pay_amt2,pay_amt3,pay_amt4,pay_amt5,pay_amt6,default_payment_next_month
0,1,20000.0,2.0,2.0,1.0,24.0,2.0,2.0,-1.0,-1.0,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1.0
1,2,120000.0,NaN,2.0,2.0,26.0,-1.0,2.0,0.0,0.0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1.0
2,3,NaN,2.0,2.0,2.0,34.0,0.0,0.0,0.0,0.0,...,14331.0,14948.0,15549.0,NaN,1500.0,1000.0,1000.0,1000.0,5000.0,0.0
3,4,50000.0,2.0,2.0,1.0,37.0,0.0,0.0,0.0,NaN,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0.0
4,5,50000.0,1.0,2.0,1.0,57.0,-1.0,0.0,-1.0,0.0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,NaN


In [ ]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

# Numerische Spalten für die modellbasierte Imputation auswählen
numeric_df = df.select_dtypes(include="number")

# Numerische Werte vor der distanzbasierten Imputation skalieren
scaler = StandardScaler()
scaled_values = scaler.fit_transform(numeric_df)

# Fehlende Werte mithilfe ähnlicher Beobachtungen imputieren
imputer = KNNImputer(n_neighbors=5)
imputed_scaled = imputer.fit_transform(scaled_values)

# Die imputierten Daten zurück in die ursprüngliche Skalierung transformieren
imputed_values = scaler.inverse_transform(imputed_scaled)

df_imputed = pd.DataFrame(
    imputed_values,
    columns=numeric_df.columns
)

# Prüfen, ob noch fehlende Werte vorhanden sind
print(df_imputed.isna().sum())

id                            0
limit_bal                     0
sex                           0
education                     0
marriage                      0
age                           0
pay_0                         0
pay_2                         0
pay_3                         0
pay_4                         0
pay_5                         0
pay_6                         0
bill_amt1                     0
bill_amt2                     0
bill_amt3                     0
bill_amt4                     0
bill_amt5                     0
bill_amt6                     0
pay_amt1                      0
pay_amt2                      0
pay_amt3                      0
pay_amt4                      0
pay_amt5                      0
pay_amt6                      0
default_payment_next_month    0
dtype: int64


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Numerische Merkmale auswählen
X_num = df.select_dtypes(include="number")

# Verbleibende fehlende Werte für die Anomalieerkennung durch den Median ersetzen
X_num = X_num.fillna(X_num.median())

# Merkmale vor dem Anpassen des Modells skalieren
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_num)

# Ein unbeaufsichtigtes Modell zur Anomalieerkennung trainieren
iso_forest = IsolationForest(
    contamination=0.05,
    random_state=42
)

anomaly_labels = iso_forest.fit_predict(X_scaled)

# Modellausgabe umwandeln: -1 bedeutet Anomalie, 1 bedeutet normal
df["is_anomaly"] = anomaly_labels == -1

print(df["is_anomaly"].value_counts())

is_anomaly
False    28785
True      1515
Name: count, dtype: int64


In [ ]:
df[df["is_anomaly"]].head()

,id,limit_bal,sex,education,marriage,age,pay_0,pay_2,pay_3,pay_4,...,bill_amt5,bill_amt6,pay_amt1,pay_amt2,pay_amt3,pay_amt4,pay_amt5,pay_amt6,default_payment_next_month,is_anomaly
6,7,500000.0,1.0,1.0,2.0,29.0,0.0,0.0,0.0,0.0,...,483003.0,473944.0,55000.0,40000.0,38000.0,20239.0,13750.0,13770.0,1.0,True
17,18,320000.0,1.0,1.0,1.0,49.0,0.0,0.0,0.0,NaN,...,5856.0,195599.0,10358.0,10000.0,75940.0,20000.0,195599.0,50000.0,0.0,True
32,33,1000000.0,1.0,1.0,2.0,32.0,0.0,0.0,0.0,0.0,...,787030.0,755890.0,30230.0,35110.0,33020.0,32040.0,32000.0,25040.0,0.0,True
33,34,500000.0,2.0,2.0,1.0,54.0,-2.0,-2.0,-2.0,-2.0,...,71439.0,8981.0,4152.0,22827.0,7521.0,71439.0,981.0,51582.0,0.0,True
40,41,360000.0,1.0,1.0,2.0,33.0,0.0,0.0,0.0,0.0,...,195969.0,179224.0,NaN,7000.0,6000.0,188840.0,28000.0,4000.0,0.0,True


In [ ]:
df[df["is_anomaly"]].sample(10, random_state=42)

,id,limit_bal,sex,education,marriage,age,pay_0,pay_2,pay_3,pay_4,...,bill_amt5,bill_amt6,pay_amt1,pay_amt2,pay_amt3,pay_amt4,pay_amt5,pay_amt6,default_payment_next_month,is_anomaly
862,863,30000.0,2.0,3.0,2.0,52.0,2.0,2.0,7.0,7.0,...,2450.0,2450.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,True
3102,3103,360000.0,2.0,2.0,2.0,41.0,0.0,0.0,0.0,0.0,...,2475.0,0.0,50000.0,100000.0,0.0,0.0,NaN,0.0,1.0,True
29649,29650,230000.0,1.0,1.0,2.0,34.0,0.0,-1.0,0.0,0.0,...,NaN,226265.0,231788.0,11621.0,9027.0,9015.0,9000.0,9014.0,NaN,True
19633,19634,500000.0,2.0,3.0,1.0,NaN,2.0,NaN,0.0,0.0,...,336073.0,1257.0,79113.0,255005.0,6780.0,8401.0,1262.0,2120.0,1.0,True
8428,8429,150000.0,1.0,1.0,2.0,35.0,1.0,2.0,2.0,2.0,...,152044.0,136643.0,6000.0,4500.0,5100.0,5322.0,20.0,5033.0,0.0,True
22878,22879,290000.0,2.0,2.0,2.0,34.0,2.0,2.0,2.0,2.0,...,227226.0,241556.0,0.0,18000.0,10000.0,0.0,18000.0,0.0,1.0,True
12328,12329,490000.0,1.0,1.0,1.0,57.0,0.0,0.0,0.0,0.0,...,303047.0,331606.0,17035.0,20059.0,12943.0,15049.0,50017.0,20021.0,0.0,True
4060,4061,320000.0,2.0,2.0,2.0,24.0,-1.0,-1.0,-1.0,0.0,...,71244.0,35299.0,51334.0,54726.0,103.0,71404.0,35448.0,47699.0,0.0,True
29628,29629,1500000.0,1.0,2.0,2.0,29.0,0.0,0.0,0.0,0.0,...,1003390.0,1025670.0,50000.0,43800.0,85280.0,50000.0,40000.0,50000.0,0.0,True
29171,29172,510000.0,NaN,3.0,1.0,61.0,0.0,0.0,0.0,2.0,...,178179.0,223100.0,8500.0,17000.0,0.0,6508.0,50000.0,7000.0,0.0,True


In [ ]:
df.to_excel("chapter5_anomaly_results.xlsx", index=False)

In [ ]:
from google.colab import files
files.download("chapter5_anomaly_results.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>